# Merge & Decode Facebook Messages
Reads all `message_*.json` files from `Data/PrivateData/`, merges them, sorts by `timestamp_ms`, fixes the Facebook Latin-1-escaped UTF-8 encoding, and writes the result to `Data/EncodedData/messages.json`.

In [57]:
import json
import glob
import os
import base64
import hashlib
import getpass
from pathlib import Path
from cryptography.fernet import Fernet

In [58]:
def fix_encoding(obj):
    """Recursively fix Facebook's mojibake: latin-1 bytes re-encoded as UTF-8."""
    if isinstance(obj, str):
        try:
            return obj.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return obj
    if isinstance(obj, list):
        return [fix_encoding(item) for item in obj]
    if isinstance(obj, dict):
        return {fix_encoding(k): fix_encoding(v) for k, v in obj.items()}
    return obj

In [59]:
base_dir = Path(os.getcwd()).parent  # messagesAnalysis/
input_dir = base_dir / 'Data' / 'PrivateData'
output_dir = base_dir / 'Data' / 'EncodedData'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Input : {input_dir}")
print(f"Output: {output_dir}")

Input : /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/PrivateData
Output: /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/EncodedData


In [60]:
input_files = sorted(input_dir.glob('message_*.json'))
print(f"Found {len(input_files)} file(s): {[f.name for f in input_files]}")

all_messages = []
participants_set = {}

for path in input_files:
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    data = fix_encoding(data)

    for p in data.get('participants', []):
        participants_set[p['name']] = p

    all_messages.extend(data.get('messages', []))

print(f"Total messages before dedup: {len(all_messages)}")

Found 5 file(s): ['message_1.json', 'message_2.json', 'message_3.json', 'message_4.json', 'message_5.json']
Total messages before dedup: 44422


In [61]:
# Sort ascending by timestamp_ms (oldest first)
all_messages.sort(key=lambda m: m.get('timestamp_ms', 0))

merged = {
    'participants': list(participants_set.values()),
    'messages': all_messages,
}

print(f"Participants : {len(merged['participants'])}")
print(f"Total messages: {len(merged['messages'])}")
print(f"Oldest : {merged['messages'][0].get('timestamp_ms')}")
print(f"Newest : {merged['messages'][-1].get('timestamp_ms')}")

Participants : 37
Total messages: 44422
Oldest : 1768955883993
Newest : 1780454758698


In [62]:
# Sanity-check: print first 1 messages
for msg in merged['messages'][:1]:
    print(msg)

{'sender_name': 'Bế Minh Nhật', 'timestamp_ms': 1768955883993, 'content': 'Chào cả nhà ^^ mình lập group chat cho buổi Tea & Talk ngày Thứ Bảy 24/1 tới này nha!', 'is_geoblocked_for_viewer': False, 'is_unsent_image_by_messenger_kid_parent': False}


---
## Level 1 — Hash sender names
Password-derived Fernet key. Every `sender_name` / `actor` / participant `name` is replaced with its ciphertext **everywhere in the data**, including inside message content strings. A lookup table mapping original name → token is stored alongside the data so Level 2 can wrap it too.

In [63]:
def password_to_fernet_key(password: str, salt: bytes = b'philosophy_group_salt_l1') -> Fernet:
    """Derive a Fernet key from a password using PBKDF2."""
    key_bytes = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, iterations=200_000)
    return Fernet(base64.urlsafe_b64encode(key_bytes))

pwd1 = getpass.getpass('Level-1 password (sender names): ')
fernet1 = password_to_fernet_key(pwd1)
print('Level-1 key derived.')

Level-1 key derived.


In [64]:
# Build a lookup: original name -> encrypted token (one token per name, reused everywhere)
all_names = {p['name'] for p in merged['participants']}
name_to_token = {
    name: fernet1.encrypt(name.encode()).decode()
    for name in all_names
}

# Sort longest names first so a short name can't partially clobber a longer one
sorted_mapping = sorted(name_to_token.items(), key=lambda x: len(x[0]), reverse=True)

def replace_names(obj, mapping):
    """Replace ALL occurrences of every name within any string in the data."""
    if isinstance(obj, str):
        result = obj
        for name, token in mapping:
            result = result.replace(name, token)
        return result
    if isinstance(obj, list):
        return [replace_names(item, mapping) for item in obj]
    if isinstance(obj, dict):
        return {k: replace_names(v, mapping) for k, v in obj.items()}
    return obj

merged_l1 = replace_names(merged, sorted_mapping)

# Attach the lookup table so we can decrypt names later
merged_l1['_name_tokens'] = name_to_token

print(f"Replaced {len(name_to_token)} unique sender names with encrypted tokens.")

Replaced 37 unique sender names with encrypted tokens.


---
## Level 2 — Encrypt the entire JSON file
The whole Level-1 JSON bytes are encrypted with a second password-derived Fernet key and stored as a single `.enc` file.

In [65]:
def password_to_fernet_key_l2(password: str, salt: bytes = b'philosophy_group_salt_l2') -> Fernet:
    key_bytes = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, iterations=200_000)
    return Fernet(base64.urlsafe_b64encode(key_bytes))

pwd2 = getpass.getpass('Level-2 password (whole file): ')
fernet2 = password_to_fernet_key_l2(pwd2)
print('Level-2 key derived.')

Level-2 key derived.


In [66]:
l1_bytes = json.dumps(merged_l1, ensure_ascii=False).encode('utf-8')
encrypted_blob = fernet2.encrypt(l1_bytes)

l2_path = output_dir / 'messages_l2.enc'
l2_path.write_bytes(encrypted_blob)

print(f"Level-2 file written: {l2_path}  ({l2_path.stat().st_size / 1024:.1f} KB)")

Level-2 file written: /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/EncodedData/messages_l2.enc  (27234.3 KB)


---
## Verification — Decrypt both layers and check first 3 messages

In [67]:
%run messages_decoder.ipynb
restored = decode_messages()

Level-2 OK — 44422 messages, 37 participants.
Level-1 OK — 37 names restored.


In [68]:
print("First message after full decryption:\n")
for msg in restored['messages'][:1]:
    print(json.dumps(msg, ensure_ascii=False, indent=2))
    print()

First message after full decryption:

{
  "sender_name": "Bế Minh Nhật",
  "timestamp_ms": 1768955883993,
  "content": "Chào cả nhà ^^ mình lập group chat cho buổi Tea & Talk ngày Thứ Bảy 24/1 tới này nha!",
  "is_geoblocked_for_viewer": false,
  "is_unsent_image_by_messenger_kid_parent": false
}

